# PM2.5 Data Enrichment Pipeline V4 (2023-2025)

This notebook creates a unified enriched dataset combining PM2.5, traffic, and weather data for the period 2023-2025.

**Version**: V4 - Multi-year analysis with selected features
**Date Range**: June 2023 - June 2025

## 1. Setup and Configuration

In [1]:
import pandas as pd
import numpy as np
import h3
from datetime import datetime, timedelta
from tqdm import tqdm
import warnings
import os
from joblib import Parallel, delayed
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

print("Libraries loaded successfully")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

Libraries loaded successfully
Pandas version: 2.3.1
NumPy version: 2.2.6


In [ ]:
DATA_PATH = '/Users/vojtech/Code/Bard89/Project-Data/data/processed/'
OUTPUT_PATH = '/Users/vojtech/Code/Bard89/smoglens-data/'
H3_RESOLUTION = 7

START_DATE = '2023-06-01'
END_DATE = '2025-06-30'

print(f"Data path: {DATA_PATH}")
print(f"Output path: {OUTPUT_PATH}")
print(f"H3 Resolution: {H3_RESOLUTION} (approx 5.16 km² per hexagon)")
print(f"Date Range: {START_DATE} to {END_DATE}")

## 2. Load Datasets

In [3]:
print("Loading OpenAQ PM2.5 data...")
df_openaq = pd.read_csv(f"{DATA_PATH}jp_openaq_processed_20230601_to_20250630.csv")
df_openaq['timestamp'] = pd.to_datetime(df_openaq['timestamp'])

selected_openaq_cols = [
    'timestamp', 'h3_index_res8', 'h3_lat_res8', 'h3_lon_res8',
    'pm25_ugm3_mean', 'pm25_ugm3_min', 'pm25_ugm3_max', 'pm25_ugm3_count'
]
df_openaq = df_openaq[selected_openaq_cols]

print(f"OpenAQ data loaded: {len(df_openaq):,} records")
print(f"Date range: {df_openaq['timestamp'].min()} to {df_openaq['timestamp'].max()}")
print(f"PM2.5 coverage: {df_openaq['pm25_ugm3_mean'].notna().mean():.1%}")

Loading OpenAQ PM2.5 data...
OpenAQ data loaded: 9,579,181 records
Date range: 2023-07-14 16:00:00+00:00 to 2025-07-26 05:00:00+00:00
PM2.5 coverage: 80.0%


In [4]:
print("\nLoading JARTIC traffic data...")
df_jartic = pd.read_csv(f"{DATA_PATH}jp_jartic_processed_20230601_to_20250630_full.csv")
df_jartic['timestamp'] = pd.to_datetime(df_jartic['timestamp'])

selected_jartic_cols = [
    'timestamp', 'h3_index_res8', 'h3_lat_res8', 'h3_lon_res8',
    'avg_traffic_volume', 'max_traffic_volume', 'avg_speed_kmh',
    'congestion_index', 'unique_links', 'measurement_count'
]
df_jartic = df_jartic[selected_jartic_cols]
df_jartic.rename(columns={'measurement_count': 'traffic_measurement_count'}, inplace=True)

print(f"JARTIC data loaded: {len(df_jartic):,} records")
print(f"Date range: {df_jartic['timestamp'].min()} to {df_jartic['timestamp'].max()}")
print(f"Traffic coverage: {df_jartic['avg_traffic_volume'].notna().mean():.1%}")


Loading JARTIC traffic data...
JARTIC data loaded: 18,849,324 records
Date range: 2023-04-30 15:00:00+00:00 to 2025-06-30 14:00:00+00:00
Traffic coverage: 100.0%


In [5]:
print("\nLoading OpenMeteo weather data...")
df_openmeteo = pd.read_csv(f"{DATA_PATH}jp_openmeteo_processed_20230601_to_20250630.csv")
df_openmeteo['timestamp'] = pd.to_datetime(df_openmeteo['timestamp'], format='mixed')

selected_weather_cols = [
    'timestamp', 'h3_index_res8', 'h3_lat_res8', 'h3_lon_res8',
    'temperature_c_mean', 'humidity_pct_mean', 'precipitation_mm_mean',
    'pressure_hpa_mean', 'cloud_cover_pct_mean', 'dew_point_c_mean',
    'solar_radiation_wm2_mean'
]
df_openmeteo = df_openmeteo[selected_weather_cols]

print(f"OpenMeteo data loaded: {len(df_openmeteo):,} records")
print(f"Date range: {df_openmeteo['timestamp'].min()} to {df_openmeteo['timestamp'].max()}")
print(f"Weather coverage: {df_openmeteo['temperature_c_mean'].notna().mean():.1%}")


Loading OpenMeteo weather data...
OpenMeteo data loaded: 547,920 records
Date range: 2023-06-01 00:00:00+00:00 to 2025-06-30 23:00:00+00:00
Weather coverage: 100.0%


## 3. Create Hexagon Registries

In [6]:
def create_hexagon_registry(df, name):
    registry = {}
    unique_hexes = df[['h3_index_res8', 'h3_lat_res8', 'h3_lon_res8']].drop_duplicates()
    
    for _, row in tqdm(unique_hexes.iterrows(), total=len(unique_hexes), desc=f"Processing {name} hexagons"):
        if pd.notna(row['h3_index_res8']):
            hex7 = h3.cell_to_parent(row['h3_index_res8'], H3_RESOLUTION)
            
            if hex7 not in registry:
                registry[hex7] = {
                    'hex7_id': hex7,
                    'center_lat': row['h3_lat_res8'],
                    'center_lon': row['h3_lon_res8'],
                    'res8_hexagons': []
                }
            
            registry[hex7]['res8_hexagons'].append(row['h3_index_res8'])
    
    return registry

print("Creating hexagon registries at resolution 7...")
pm25_registry = create_hexagon_registry(df_openaq, "PM2.5")
traffic_registry = create_hexagon_registry(df_jartic, "Traffic")
weather_registry = create_hexagon_registry(df_openmeteo, "Weather")

print(f"\nRegistry sizes:")
print(f"  PM2.5: {len(pm25_registry)} hexagons")
print(f"  Traffic: {len(traffic_registry)} hexagons")
print(f"  Weather: {len(weather_registry)} hexagons")

Creating hexagon registries at resolution 7...


Processing Weather hexagons: 100%|██████████| 30/30 [00:00<00:00, 45311.17it/s]


Registry sizes:
  PM2.5: 634 hexagons
  Traffic: 1018 hexagons
  Weather: 30 hexagons


## 4. Build Nearest Neighbor Lookup

In [7]:
def haversine_distance(lat1, lon1, lat2, lon2):
    from math import radians, cos, sin, asin, sqrt
    
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a))
    r = 6371
    
    return c * r

def find_k_nearest_hexagons(target_hex, target_lat, target_lon, source_registry, k=3, max_distance_km=500):
    distances = []
    
    for source_hex, source_info in source_registry.items():
        if source_hex == target_hex:
            return [source_hex], [0.0]
        
        distance = haversine_distance(target_lat, target_lon, 
                                     source_info['center_lat'], 
                                     source_info['center_lon'])
        
        if distance < max_distance_km:
            distances.append((source_hex, distance))
    
    distances.sort(key=lambda x: x[1])
    
    nearest_hexes = [h for h, _ in distances[:k]]
    nearest_distances = [d for _, d in distances[:k]]
    
    return nearest_hexes, nearest_distances

In [8]:
print("Building nearest neighbor lookup tables...")
print("This may take a few minutes...")

nearest_lookup = {}

for hex7, hex_info in tqdm(pm25_registry.items(), desc="Processing PM2.5 hexagons"):
    lat = hex_info['center_lat']
    lon = hex_info['center_lon']
    
    traffic_hexes, traffic_distances = find_k_nearest_hexagons(hex7, lat, lon, traffic_registry, k=3)
    weather_hexes, weather_distances = find_k_nearest_hexagons(hex7, lat, lon, weather_registry, k=3)
    
    nearest_lookup[hex7] = {
        'nearest_traffic_hexes': traffic_hexes,
        'traffic_distances_km': traffic_distances,
        'has_local_traffic': len(traffic_distances) > 0 and traffic_distances[0] == 0,
        'nearest_weather_hexes': weather_hexes,
        'weather_distances_km': weather_distances,
        'has_local_weather': len(weather_distances) > 0 and weather_distances[0] == 0
    }

nearest_df = pd.DataFrame(nearest_lookup).T.reset_index()
nearest_df.rename(columns={'index': 'hex7_id'}, inplace=True)

print(f"\nNearest neighbor statistics:")
print(f"  Hexagons with local traffic: {nearest_df['has_local_traffic'].sum()}")
print(f"  Hexagons with local weather: {nearest_df['has_local_weather'].sum()}")

Building nearest neighbor lookup tables...
This may take a few minutes...


Processing PM2.5 hexagons: 100%|██████████| 634/634 [00:00<00:00, 1149.65it/s]


Nearest neighbor statistics:
  Hexagons with local traffic: 26
  Hexagons with local weather: 7


## 5. Aggregate Data to Resolution 7

In [9]:
print("Aggregating PM2.5 data to resolution 7...")

df_openaq['hex7_id'] = df_openaq['h3_index_res8'].apply(
    lambda x: h3.cell_to_parent(x, H3_RESOLUTION) if pd.notna(x) else None
)
df_openaq['hour'] = df_openaq['timestamp'].dt.floor('H')

pm25_hourly = df_openaq.groupby(['hex7_id', 'hour']).agg({
    'pm25_ugm3_mean': 'mean',
    'pm25_ugm3_min': 'min',
    'pm25_ugm3_max': 'max',
    'pm25_ugm3_count': 'sum',
    'h3_lat_res8': 'mean',
    'h3_lon_res8': 'mean'
}).reset_index()

pm25_hourly.rename(columns={
    'hour': 'timestamp',
    'h3_lat_res8': 'lat',
    'h3_lon_res8': 'lon'
}, inplace=True)

print(f"Created {len(pm25_hourly):,} hourly PM2.5 records for {pm25_hourly['hex7_id'].nunique()} hexagons")

Aggregating PM2.5 data to resolution 7...
Created 9,457,112 hourly PM2.5 records for 634 hexagons


In [10]:
print("\nAggregating traffic data to resolution 7...")

df_jartic['hex7_id'] = df_jartic['h3_index_res8'].apply(
    lambda x: h3.cell_to_parent(x, H3_RESOLUTION) if pd.notna(x) else None
)
df_jartic['hour'] = df_jartic['timestamp'].dt.floor('H')

traffic_hourly = df_jartic.groupby(['hex7_id', 'hour']).agg({
    'avg_traffic_volume': 'mean',
    'max_traffic_volume': 'max',
    'avg_speed_kmh': 'mean',
    'congestion_index': 'mean',
    'unique_links': 'sum',
    'traffic_measurement_count': 'sum'
}).reset_index()

traffic_hourly.rename(columns={'hour': 'timestamp'}, inplace=True)

print(f"Created {len(traffic_hourly):,} hourly traffic records for {traffic_hourly['hex7_id'].nunique()} hexagons")


Aggregating traffic data to resolution 7...
Created 18,849,324 hourly traffic records for 1018 hexagons


In [11]:
print("\nAggregating weather data to resolution 7...")

df_openmeteo['hex7_id'] = df_openmeteo['h3_index_res8'].apply(
    lambda x: h3.cell_to_parent(x, H3_RESOLUTION) if pd.notna(x) else None
)
df_openmeteo['hour'] = df_openmeteo['timestamp'].dt.floor('H')

weather_hourly = df_openmeteo.groupby(['hex7_id', 'hour']).agg({
    'temperature_c_mean': 'mean',
    'humidity_pct_mean': 'mean',
    'precipitation_mm_mean': 'mean',
    'pressure_hpa_mean': 'mean',
    'cloud_cover_pct_mean': 'mean',
    'dew_point_c_mean': 'mean',
    'solar_radiation_wm2_mean': 'mean'
}).reset_index()

weather_hourly.rename(columns={'hour': 'timestamp'}, inplace=True)

print(f"Created {len(weather_hourly):,} hourly weather records for {weather_hourly['hex7_id'].nunique()} hexagons")


Aggregating weather data to resolution 7...
Created 547,920 hourly weather records for 30 hexagons


## 6. Merge and Enrich Data

In [12]:
print("Creating enriched dataset...")

enriched_data = pm25_hourly.copy()

enriched_data = enriched_data.merge(nearest_df, on='hex7_id', how='left')

print(f"Starting with {len(enriched_data):,} PM2.5 records")

Creating enriched dataset...
Starting with 9,457,112 PM2.5 records


In [13]:
print("Creating lookup dictionaries for faster processing...")

traffic_lookup = {}
for _, row in traffic_hourly.iterrows():
    key = (row['hex7_id'], row['timestamp'])
    traffic_lookup[key] = {
        'avg_traffic_volume': row['avg_traffic_volume'],
        'max_traffic_volume': row['max_traffic_volume'],
        'avg_speed_kmh': row['avg_speed_kmh'],
        'congestion_index': row['congestion_index'],
        'unique_links': row['unique_links'],
        'traffic_measurement_count': row['traffic_measurement_count']
    }

weather_lookup = {}
for _, row in weather_hourly.iterrows():
    key = (row['hex7_id'], row['timestamp'])
    weather_lookup[key] = {
        'temperature_c_mean': row['temperature_c_mean'],
        'humidity_pct_mean': row['humidity_pct_mean'],
        'precipitation_mm_mean': row['precipitation_mm_mean'],
        'pressure_hpa_mean': row['pressure_hpa_mean'],
        'cloud_cover_pct_mean': row['cloud_cover_pct_mean'],
        'dew_point_c_mean': row['dew_point_c_mean'],
        'solar_radiation_wm2_mean': row['solar_radiation_wm2_mean']
    }

print(f"Created lookups: {len(traffic_lookup):,} traffic, {len(weather_lookup):,} weather entries")

Creating lookup dictionaries for faster processing...
Created lookups: 18,849,324 traffic, 547,920 weather entries


In [14]:
def process_row_with_knn(row_data):
    idx, row = row_data
    
    traffic_result = {}
    if row['has_local_traffic']:
        key = (row['hex7_id'], row['timestamp'])
        if key in traffic_lookup:
            traffic_result = traffic_lookup[key].copy()
            traffic_result['traffic_distance_km'] = 0.0
    else:
        traffic_values = {}
        weights = []
        
        for hex_id, distance in zip(row['nearest_traffic_hexes'], row['traffic_distances_km']):
            key = (hex_id, row['timestamp'])
            if key in traffic_lookup:
                weight = 1 / (1 + distance)
                weights.append(weight)
                
                for feature, value in traffic_lookup[key].items():
                    if feature not in traffic_values:
                        traffic_values[feature] = []
                    traffic_values[feature].append(value * weight)
        
        if weights:
            total_weight = sum(weights)
            for feature in traffic_values:
                traffic_result[feature] = sum(traffic_values[feature]) / total_weight
            traffic_result['traffic_distance_km'] = row['traffic_distances_km'][0] if row['traffic_distances_km'] else np.nan
        else:
            for feature in ['avg_traffic_volume', 'max_traffic_volume', 'avg_speed_kmh', 
                          'congestion_index', 'unique_links', 'traffic_measurement_count']:
                traffic_result[feature] = np.nan
            traffic_result['traffic_distance_km'] = np.nan
    
    weather_result = {}
    if row['has_local_weather']:
        key = (row['hex7_id'], row['timestamp'])
        if key in weather_lookup:
            weather_result = weather_lookup[key].copy()
            weather_result['weather_distance_km'] = 0.0
    else:
        weather_values = {}
        weights = []
        
        for hex_id, distance in zip(row['nearest_weather_hexes'], row['weather_distances_km']):
            key = (hex_id, row['timestamp'])
            if key in weather_lookup:
                weight = 1 / (1 + distance)
                weights.append(weight)
                
                for feature, value in weather_lookup[key].items():
                    if feature not in weather_values:
                        weather_values[feature] = []
                    weather_values[feature].append(value * weight)
        
        if weights:
            total_weight = sum(weights)
            for feature in weather_values:
                weather_result[feature] = sum(weather_values[feature]) / total_weight
            weather_result['weather_distance_km'] = row['weather_distances_km'][0] if row['weather_distances_km'] else np.nan
        else:
            for feature in ['temperature_c_mean', 'humidity_pct_mean', 'precipitation_mm_mean',
                          'pressure_hpa_mean', 'cloud_cover_pct_mean', 'dew_point_c_mean',
                          'solar_radiation_wm2_mean']:
                weather_result[feature] = np.nan
            weather_result['weather_distance_km'] = np.nan
    
    combined = {**traffic_result, **weather_result}
    return idx, combined

In [15]:
print("Processing data enrichment with K-NN interpolation...")
print(f"Total records to process: {len(enriched_data):,}")
print("This may take 10-15 minutes for the full dataset...")

row_data = list(enriched_data.iterrows())

n_jobs = min(8, -1)
print(f"Using parallel processing with {n_jobs} jobs")

results = Parallel(n_jobs=n_jobs, backend='threading')(
    delayed(process_row_with_knn)(row) 
    for row in tqdm(row_data, desc="Enriching data")
)

results.sort(key=lambda x: x[0])

enrichment_data = [result[1] for result in results]
enrichment_df = pd.DataFrame(enrichment_data)

enriched_data = pd.concat([enriched_data, enrichment_df], axis=1)

columns_to_drop = ['nearest_traffic_hexes', 'traffic_distances_km', 
                  'nearest_weather_hexes', 'weather_distances_km',
                  'has_local_traffic', 'has_local_weather']
enriched_data = enriched_data.drop(columns=columns_to_drop, errors='ignore')

print(f"\n✓ Enrichment complete! Dataset shape: {enriched_data.shape}")

Processing data enrichment with K-NN interpolation...
Total records to process: 9,457,112
This may take 10-15 minutes for the full dataset...
Using parallel processing with -1 jobs


Enriching data: 100%|██████████| 9457112/9457112 [05:36<00:00, 28110.98it/s]



✓ Enrichment complete! Dataset shape: (9457112, 23)


## 7. Add Engineered Features

In [16]:
print("Adding temporal features...")

enriched_data['timestamp'] = pd.to_datetime(enriched_data['timestamp'])
enriched_data['hour'] = enriched_data['timestamp'].dt.hour
enriched_data['day_of_week'] = enriched_data['timestamp'].dt.dayofweek
enriched_data['month'] = enriched_data['timestamp'].dt.month
enriched_data['year'] = enriched_data['timestamp'].dt.year
enriched_data['is_weekend'] = (enriched_data['day_of_week'] >= 5).astype(int)

enriched_data['hour_sin'] = np.sin(2 * np.pi * enriched_data['hour'] / 24)
enriched_data['hour_cos'] = np.cos(2 * np.pi * enriched_data['hour'] / 24)
enriched_data['dow_sin'] = np.sin(2 * np.pi * enriched_data['day_of_week'] / 7)
enriched_data['dow_cos'] = np.cos(2 * np.pi * enriched_data['day_of_week'] / 7)
enriched_data['month_sin'] = np.sin(2 * np.pi * enriched_data['month'] / 12)
enriched_data['month_cos'] = np.cos(2 * np.pi * enriched_data['month'] / 12)

print("Temporal features added")

Adding temporal features...
Temporal features added


In [17]:
print("Adding derived features...")

enriched_data['pm25_range'] = enriched_data['pm25_ugm3_max'] - enriched_data['pm25_ugm3_min']

def calculate_heat_index(temp_c, humidity_pct):
    temp_f = temp_c * 9/5 + 32
    
    if temp_f < 80:
        return temp_c
    
    hi = -42.379 + 2.04901523*temp_f + 10.14333127*humidity_pct - \
         0.22475541*temp_f*humidity_pct - 0.00683783*temp_f**2 - \
         0.05481717*humidity_pct**2 + 0.00122874*temp_f**2*humidity_pct + \
         0.00085282*temp_f*humidity_pct**2 - 0.00000199*temp_f**2*humidity_pct**2
    
    hi_c = (hi - 32) * 5/9
    return hi_c

enriched_data['heat_index'] = enriched_data.apply(
    lambda row: calculate_heat_index(row['temperature_c_mean'], row['humidity_pct_mean']) 
    if pd.notna(row['temperature_c_mean']) and pd.notna(row['humidity_pct_mean']) else np.nan,
    axis=1
)

enriched_data['is_raining'] = (enriched_data['precipitation_mm_mean'] > 0).astype(int)

traffic_percentile = enriched_data['avg_traffic_volume'].quantile([0.25, 0.75])
enriched_data['traffic_intensity'] = pd.cut(
    enriched_data['avg_traffic_volume'],
    bins=[-np.inf, traffic_percentile[0.25], traffic_percentile[0.75], np.inf],
    labels=['low', 'medium', 'high']
)
enriched_data['traffic_intensity'] = enriched_data['traffic_intensity'].cat.codes

enriched_data['data_completeness_score'] = (
    enriched_data[['pm25_ugm3_mean', 'avg_traffic_volume', 'temperature_c_mean']].notna().sum(axis=1) / 3
)

print("Derived features added")

Adding derived features...
Derived features added


## 8. Save Enriched Dataset

In [18]:
timestamp_str = datetime.now().strftime('%Y%m%d_%H%M%S')
output_file = f"{OUTPUT_PATH}pm25_enriched_2023_2025_v4_{timestamp_str}.csv"

print(f"Saving enriched dataset to CSV...")
print(f"Output file: {output_file}")

column_order = [
    'timestamp', 'hex7_id', 'lat', 'lon',
    'pm25_ugm3_mean', 'pm25_ugm3_min', 'pm25_ugm3_max', 'pm25_ugm3_count', 'pm25_range',
    'avg_traffic_volume', 'max_traffic_volume', 'avg_speed_kmh', 'congestion_index', 
    'unique_links', 'traffic_measurement_count', 'traffic_distance_km', 'traffic_intensity',
    'temperature_c_mean', 'humidity_pct_mean', 'precipitation_mm_mean', 'pressure_hpa_mean',
    'cloud_cover_pct_mean', 'dew_point_c_mean', 'solar_radiation_wm2_mean', 
    'weather_distance_km', 'heat_index', 'is_raining',
    'hour', 'day_of_week', 'month', 'year', 'is_weekend',
    'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos',
    'data_completeness_score'
]

available_columns = [col for col in column_order if col in enriched_data.columns]
enriched_data_final = enriched_data[available_columns]

chunk_size = 500000
n_chunks = (len(enriched_data_final) + chunk_size - 1) // chunk_size

for i in tqdm(range(n_chunks), desc="Saving CSV chunks"):
    start_idx = i * chunk_size
    end_idx = min((i + 1) * chunk_size, len(enriched_data_final))
    chunk = enriched_data_final.iloc[start_idx:end_idx]
    
    mode = 'w' if i == 0 else 'a'
    header = i == 0
    
    chunk.to_csv(output_file, mode=mode, header=header, index=False)

file_size_mb = os.path.getsize(output_file) / 1024 / 1024
print(f"\n✓ CSV saved! File size: {file_size_mb:.1f} MB")

Saving enriched dataset to CSV...
Output file: /Users/vojtech/Code/Bard89/smoglens-02/voi/V4_enrichement_OPEANAQ_OPENMETEO_JARTIC/pm25_enriched_2023_2025_v4_20250830_222050.csv


Saving CSV chunks: 100%|██████████| 19/19 [02:07<00:00,  6.71s/it]


✓ CSV saved! File size: 4372.0 MB


## 9. Summary Statistics

In [19]:
print("="*80)
print("PM2.5 ENRICHMENT PIPELINE V4 COMPLETE")
print("="*80)
print(f"\nDataset Summary:")
print(f"  Total records: {len(enriched_data_final):,}")
print(f"  Total features: {len(enriched_data_final.columns)}")
print(f"  Date range: {enriched_data_final['timestamp'].min()} to {enriched_data_final['timestamp'].max()}")
print(f"  Unique hexagons: {enriched_data_final['hex7_id'].nunique()}")
print(f"  File size: {file_size_mb:.1f} MB")

print(f"\nData Completeness:")
for col in ['pm25_ugm3_mean', 'avg_traffic_volume', 'temperature_c_mean']:
    if col in enriched_data_final.columns:
        completeness = enriched_data_final[col].notna().mean()
        print(f"  {col}: {completeness:.1%}")

print(f"\nOutput file: {output_file}")
print("="*80)

PM2.5 ENRICHMENT PIPELINE V4 COMPLETE

Dataset Summary:
  Total records: 9,457,112
  Total features: 39
  Date range: 2023-07-14 16:00:00+00:00 to 2025-07-26 05:00:00+00:00
  Unique hexagons: 634
  File size: 4372.0 MB

Data Completeness:
  pm25_ugm3_mean: 80.5%
  avg_traffic_volume: 96.3%
  temperature_c_mean: 96.1%

Output file: /Users/vojtech/Code/Bard89/smoglens-02/voi/V4_enrichement_OPEANAQ_OPENMETEO_JARTIC/pm25_enriched_2023_2025_v4_20250830_222050.csv


In [20]:
print("\nFeature Statistics:")
numeric_cols = enriched_data_final.select_dtypes(include=[np.number]).columns
enriched_data_final[numeric_cols].describe()


Feature Statistics:


,lat,lon,pm25_ugm3_mean,pm25_ugm3_min,pm25_ugm3_max,pm25_ugm3_count,pm25_range,avg_traffic_volume,max_traffic_volume,avg_speed_kmh,congestion_index,unique_links,traffic_measurement_count,traffic_distance_km,traffic_intensity,temperature_c_mean,humidity_pct_mean,precipitation_mm_mean,pressure_hpa_mean,cloud_cover_pct_mean,dew_point_c_mean,solar_radiation_wm2_mean,weather_distance_km,heat_index,is_raining,hour,day_of_week,month,year,is_weekend,hour_sin,hour_cos,dow_sin,dow_cos,month_sin,month_cos,data_completeness_score
count,9457112.000,9457112.000,7617538.000,7617538.000,7617538.000,9457112.000,7617538.000,9105653.000,9105653.000,9105653.000,9105653.000,9105653.000,9105653.000,9105653.000,9457112.000,9091320.000,9091320.000,9091320.000,9091320.000,9091320.000,9091320.000,9060846.000,9091320.000,9091320.000,9457112.000,9457112.000,9457112.000,9457112.000,9457112.000,9457112.000,9457112.000,9457112.000,9457112.000,9457112.000,9457112.000,9457112.000,9457112.000
mean,35.475,136.655,8.999,8.986,9.011,0.812,0.024,34.777,109.977,42.617,7.327,72.259,836.541,5.909,0.926,15.795,75.264,0.194,1011.950,59.603,11.100,84.963,38.072,16.355,0.283,11.342,2.995,6.575,2024.070,0.285,0.033,0.083,0.001,0.001,-0.021,-0.015,0.910
std,2.152,3.563,7.987,7.988,7.990,0.408,0.377,21.906,70.605,10.939,2.873,80.905,920.736,14.303,0.790,9.131,15.026,0.771,7.117,38.946,9.821,161.932,36.864,10.064,0.451,7.200,2.000,3.413,0.706,0.451,0.728,0.679,0.707,0.707,0.705,0.709,0.182
min,24.339,124.159,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1.000,1.000,0.000,-1.000,-20.900,10.000,0.000,978.000,0.000,-23.100,0.000,0.000,-20.900,0.000,0.000,0.000,1.000,2023.000,0.000,-1.000,-1.000,-0.975,-0.901,-1.000,-1.000,0.000
25%,34.483,133.961,5.000,5.000,5.000,1.000,0.000,12.900,55.911,33.941,5.380,16.380,189.914,3.104,0.000,8.099,65.296,0.000,1007.008,18.759,2.694,0.000,10.937,8.099,0.000,5.000,1.000,4.000,2024.000,0.000,-0.707,-0.500,-0.782,-0.901,-0.866,-0.866,1.000
50%,35.250,136.916,8.000,8.000,8.000,1.000,0.000,36.668,100.586,41.666,8.416,41.845,496.332,4.246,1.000,16.170,77.870,0.000,1011.737,70.183,11.876,0.000,25.068,16.170,0.000,11.000,3.000,7.000,2024.000,0.000,0.000,0.000,0.000,-0.223,-0.000,-0.000,1.000
75%,35.924,139.719,12.000,12.000,12.000,1.000,0.000,52.119,154.428,53.550,10.000,100.061,1173.610,5.546,1.000,23.655,87.442,0.032,1017.071,99.742,20.375,87.052,56.229,23.655,1.000,18.000,5.000,10.000,2025.000,1.000,0.707,0.707,0.782,0.623,0.500,0.500,1.000
max,45.120,144.369,4170.000,4170.000,4170.000,2.000,115.000,235.333,5132.227,60.000,10.000,527.690,6194.432,406.759,2.000,38.024,100.000,45.558,1034.880,100.000,29.112,855.251,245.556,49.343,1.000,23.000,6.000,12.000,2025.000,1.000,1.000,1.000,0.975,1.000,1.000,1.000,1.000
